# Compute standard deviation using CUDA reduction.

In [2]:
import numpy as np
from numba import cuda
import math

# =====================================================
# Generate HIGGS-like Numerical Data
# =====================================================

data = np.random.normal(
    loc=0,
    scale=1,
    size=1_000_000
).astype(np.float32)

print("Samples Loaded:", len(data))

# =====================================================
# Compute Mean
# =====================================================

mean = np.mean(data).astype(np.float32)

# =====================================================
# CUDA Kernel
# =====================================================

@cuda.jit
def squared_diff_kernel(data, mean, output):

    idx = cuda.grid(1)

    if idx < data.size:
        diff = data[idx] - mean
        output[idx] = diff * diff

# =====================================================
# Allocate GPU Memory
# =====================================================

d_data = cuda.to_device(data)
d_output = cuda.device_array_like(data)

threads_per_block = 256

blocks_per_grid = (
    data.size + threads_per_block - 1
) // threads_per_block

# =====================================================
# Launch Kernel
# =====================================================

squared_diff_kernel[
    blocks_per_grid,
    threads_per_block
](
    d_data,
    mean,
    d_output
)

cuda.synchronize()

# =====================================================
# Reduction
# =====================================================

sq_diff = d_output.copy_to_host()

variance = np.sum(sq_diff) / len(data)

std_dev = math.sqrt(variance)

# =====================================================
# Results
# =====================================================

print("Mean               :", mean)
print("Variance           :", variance)
print("Standard Deviation :", std_dev)

Samples Loaded: 1000000
Mean               : 0.0015396115
Variance           : 1.0005293
Standard Deviation : 1.000264609613679
